# India Digital Payments: Data Preparation & SQL

**Author: Mansi Saxena**

[LinkedIn](https://www.linkedin.com/in/mansi-saxena-049102362) | [GitHub](https://github.com/050804mansi) | [Email](mailto:mansi53245@gmail.com)

Load RBI annual payment-system tables, inspect units and data quality, then load the raw columns into a local SQLite database. Source and field definitions are in `DATA_SOURCES.md`.

## 1. Load and inspect source tables

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'Data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'Data'
modes = pd.read_csv(DATA / 'annual_payment_mode_statistics.csv')
totals = pd.read_csv(DATA / 'annual_payment_system_totals.csv')
print('Mode rows:', modes.shape, '| Aggregate rows:', totals.shape)
display(modes.head())
display(totals)

Mode rows: (35, 5) | Aggregate rows: (5, 5)


,Financial_Year,Payment_Mode,Mode_Group,Transaction_Volume_Lakh,Transaction_Value_Lakh_Crore
0,2021-22,RTGS,Large_Value,2078,1286.6
1,2022-23,RTGS,Large_Value,2426,1499.5
2,2023-24,RTGS,Large_Value,2700,1708.9
3,2024-25,RTGS,Large_Value,3025,2013.9
4,2025-26,RTGS,Large_Value,3500,2273.7


,Financial_Year,Total_Retail_Volume_Lakh,Total_Retail_Value_Lakh_Crore,Total_Digital_Volume_Lakh,Total_Digital_Value_Lakh_Crore
0,2021-22,724689,523.9,719768,1744.0
1,2022-23,1144065,659.1,1139382,2086.8
2,2023-24,1648234,791.5,1644302,2428.2
3,2024-25,2222885,920.3,2219815,2863.0
4,2025-26,2819536,1048.8,2817448,3251.0


## 2. Check categories, missing values and units

In [2]:
print('Years:', sorted(modes.Financial_Year.unique()))
print('Modes:', sorted(modes.Payment_Mode.unique()))
print('Missing values by field:')
display(modes.isna().sum().to_frame('Missing'))
assert not modes.isna().any().any()
assert not totals.isna().any().any()
assert modes.groupby('Payment_Mode').Financial_Year.nunique().eq(5).all()

Years: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Modes: ['Credit_Cards', 'Debit_Cards', 'IMPS', 'NEFT', 'Prepaid_Instruments', 'RTGS', 'UPI']
Missing values by field:


,Missing
Financial_Year,0
Payment_Mode,0
Mode_Group,0
Transaction_Volume_Lakh,0
Transaction_Value_Lakh_Crore,0


## 3. Create useful derived metrics

In [3]:
modes = modes.sort_values(['Payment_Mode', 'Financial_Year']).copy()
modes['Estimated_Average_Ticket_INR'] = (
    modes.Transaction_Value_Lakh_Crore / modes.Transaction_Volume_Lakh * 10_000_000
)
modes['YoY_Volume_Growth_Pct'] = modes.groupby('Payment_Mode').Transaction_Volume_Lakh.pct_change() * 100
upi = modes[modes.Payment_Mode.eq('UPI')][['Financial_Year', 'Transaction_Volume_Lakh']]
totals = totals.merge(upi, on='Financial_Year', validate='one_to_one').rename(
    columns={'Transaction_Volume_Lakh': 'UPI_Volume_Lakh'}
)
totals['UPI_Share_Of_Retail_Volume_Pct'] = totals.UPI_Volume_Lakh / totals.Total_Retail_Volume_Lakh * 100
display(modes[modes.Payment_Mode.eq('UPI')][['Financial_Year', 'Estimated_Average_Ticket_INR', 'YoY_Volume_Growth_Pct']])
display(totals[['Financial_Year', 'UPI_Share_Of_Retail_Volume_Pct']])

,Financial_Year,Estimated_Average_Ticket_INR,YoY_Volume_Growth_Pct
5,2021-22,1832.183323,NaN
6,2022-23,1661.601827,82.161672
7,2023-24,1525.209812,56.639121
8,2024-25,1402.085373,41.742323
9,2025-26,1300.405725,29.995212


,Financial_Year,UPI_Share_Of_Retail_Volume_Pct
0,2021-22,63.414927
1,2022-23,73.172766
2,2023-24,79.557575
3,2024-25,83.614762
4,2025-26,85.693852


## 4. Load data into SQLite and query it

In [4]:
db_path = DATA / 'annual_payment_analytics.sqlite'
with sqlite3.connect(db_path) as con:
    modes.to_sql('annual_payment_modes', con, if_exists='replace', index=False)
    totals.to_sql('annual_payment_totals', con, if_exists='replace', index=False)
    query = '''
        SELECT Financial_Year, Payment_Mode, Transaction_Volume_Lakh,
               ROUND(Estimated_Average_Ticket_INR, 2) AS Estimated_Average_Ticket_INR
        FROM annual_payment_modes
        WHERE Payment_Mode IN ('UPI', 'NEFT', 'IMPS')
        ORDER BY Payment_Mode, Financial_Year
    '''
    display(pd.read_sql_query(query, con))
print('SQLite database:', db_path)

,Financial_Year,Payment_Mode,Transaction_Volume_Lakh,Estimated_Average_Ticket_INR
0,2021-22,IMPS,46625,8943.70
1,2022-23,IMPS,56533,9888.03
2,2023-24,IMPS,60053,10823.77
3,2024-25,IMPS,56250,12693.33
4,2025-26,IMPS,49431,15516.58
5,2021-22,NEFT,40407,71101.54
6,2022-23,NEFT,52847,63806.84
7,2023-24,NEFT,72640,53882.16
8,2024-25,NEFT,96198,46217.18
9,2025-26,NEFT,100996,49734.64


SQLite database: c:\Users\MANSI\Downloads\Digital-Payments-Channel-Analytics-Mansi-Saxena\Data\annual_payment_analytics.sqlite


## Takeaway

These are RBI-reported annual aggregates, not individual customer transactions. The next notebook compares channel growth and visualises the derived metrics.